In [1]:
import os

In [2]:
%pwd

'd:\\NLP Project\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'd:\\NLP Project'

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [7]:
from NexText.constants import *
from NexText.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

In [9]:
import os
from NexText.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

d:\NLP Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import os

from datasets import load_from_disk
from transformers import AutoTokenizer

from NexText.logging import logger


class DataTransformation:

    def __init__(self, config: DataTransformationConfig):
        self.config = config

        self.tokenizer = AutoTokenizer.from_pretrained(
            config.tokenizer_name
        )

    def convert_examples_to_features(self, example_batch):

        input_encodings = self.tokenizer(
            example_batch["dialogue"],
            max_length=512,
            truncation=True
        )

        target_encodings = self.tokenizer(
            text_target=example_batch["summary"],
            max_length=128,
            truncation=True
        )

        return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "labels": target_encodings["input_ids"]
        }

    def convert(self):

        logger.info("Loading dataset...")

        dataset_samsum = load_from_disk(
            self.config.data_path
        )

        logger.info("Tokenizing dataset...")

        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True
        )

        output_path = os.path.join(
            self.config.root_dir,
            "samsum_dataset"
        )

        dataset_samsum_pt.save_to_disk(
            output_path
        )

        logger.info(
            f"Transformed dataset saved to: {output_path}"
        )

In [13]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[ 2026-09-23 10:43:45,794 ] 21 NexText_logger - INFO - YAML file: D:\NLP Project\config\config.yaml loaded successfully.
[ 2026-09-23 10:43:45,795 ] 21 NexText_logger - INFO - YAML file: D:\NLP Project\params.yaml loaded successfully.
[ 2026-09-23 10:43:45,797 ] 49 NexText_logger - INFO - Created directory at: artifacts
[ 2026-09-23 10:43:45,797 ] 49 NexText_logger - INFO - Created directory at: artifacts/data_transformation
[ 2026-09-23 10:43:46,085 ] 1025 httpx - INFO - HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[ 2026-09-23 10:43:46,115 ] 1025 httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2Fmain%2Fconfig.json=&etag=%222c1a911e577525af99c26c1634c473667e1e7ae2%22 "HTTP/1.1 200 OK"
[ 2026-09-23 10:43:46,339 ] 1025 httpx - INFO - HTTP Request

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 43333.64 examples/s]

[ 2026-09-23 10:43:55,229 ] 62 NexText_logger - INFO - Transformed dataset saved to: artifacts/data_transformation\samsum_dataset
